In [7]:
# Standard library imports
import os
import sys

# Third-party imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# Local imports
module_path = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from paths import BASE_INPUT_PATH, BASE_OUTPUT_PATH

In [10]:
postprocessed_data_path = BASE_OUTPUT_PATH / 'postprocessed_data'

ex_statistical = pd.read_csv(postprocessed_data_path / 'expanding_window_statistical_models_comparison.csv')
rw_statistical = pd.read_csv(postprocessed_data_path / 'rolling_window_statistical_models_comparison.csv')

ex_lstm = pd.read_csv(postprocessed_data_path / 'expanding_window_lstm_models_comparison.csv')
ex_lstm_42 = ex_lstm[ex_lstm['MODEL'].str.contains('ts42')].reset_index(drop=True)
ex_lstm_168 = ex_lstm[ex_lstm['MODEL'].str.contains('ts168')].reset_index(drop=True)

# Define the path for saving plot outputs
plot_output_path = BASE_OUTPUT_PATH / 'plots'

In [9]:
new_names = {
    'rw_arma_3m': 'ARMA (3 Months)',
    'rw_arma_6m': 'ARMA (6 Months)',
    'rw_arma_12m': 'ARMA (12 Months)',
    'rw_armax_exog_1_3m': 'ARMAX (Exog. 1) (3 Months)',
    'rw_armax_exog_1_6m': 'ARMAX (Exog. 1) (6 Months)',
    'rw_armax_exog_1_12m': 'ARMAX (Exog. 1) (12 Months)',
    'rw_armax_3m': 'ARMAX (Exog. 4) (3 Months)',
    'rw_armax_6m': 'ARMAX (Exog. 4) (6 Months)',
    'rw_armax_12m': 'ARMAX (Exog. 4) (12 Months)',
    'rw_armax_exog_1_3_3m': 'ARMAX (Exog. 1&3) (3 Months)',
    'rw_armax_exog_1_3_6m': 'ARMAX (Exog. 1&3) (6 Months)',
    'rw_armax_exog_1_3_12m': 'ARMAX (Exog. 1&3) (12 Months)',
    'rw_armax_exog_1_4_3m': 'ARMAX (Exog. 1&4) (3 Months)',
    'rw_armax_exog_1_4_6m': 'ARMAX (Exog. 1&4) (6 Months)',
    'rw_armax_exog_1_4_12m': 'ARMAX (Exog. 1&4) (12 Months)',
    'rw_sarma_3m': 'SARMA (3 Months)',
    'rw_sarma_6m': 'SARMA (6 Months)',
    'rw_sarma_12m': 'SARMA (12 Months)',
    'rw_sarmax_exog_1_3m': 'SARMAX (Exog. 1) (3 Months)',
    'rw_sarmax_exog_1_6m': 'SARMAX (Exog. 1) (6 Months)',
    'rw_sarmax_exog_1_12m': 'SARMAX (Exog. 1) (12 Months)',
    # 'rw_sarmax_3m': 'SARMAX (Exog. 4) (3 Months)',
    # 'rw_sarmax_6m': 'SARMAX (Exog. 4) (6 Months)',
    # 'rw_sarmax_12m': 'SARMAX (Exog. 4) (12 Months)',
    # 'rw_sarmax_exog_1_3_3m': 'SARMAX (Exog. 1&3) (3 Months)',
    # 'rw_sarmax_exog_1_3_6m': 'SARMAX (Exog. 1&3) (6 Months)',
    # 'rw_sarmax_exog_1_3_12m': 'SARMAX (Exog. 1&3) (12 Months)',
    # 'rw_sarmax_exog_1_4_3m': 'SARMAX (Exog. 1&4) (3 Months)',
    # 'rw_sarmax_exog_1_4_6m': 'SARMAX (Exog. 1&4) (6 Months)',
    # 'rw_sarmax_exog_1_4_12m': 'SARMAX (Exog. 1&4) (12 Months)',
    'arma': 'ARMA',
    'armax_exog_1': 'ARMAX (Exog. 1)',
    'armax': 'ARMAX (Exog. 4)',
    'armax_exog_1_3': 'ARMAX (Exog. 1&3)',
    'armax_exog_1_4': 'ARMAX (Exog. 1&4)',
    'sarma': 'SARMA',
    'sarmax_exog_1': 'SARMAX (Exog. 1)',
    'sarmax': 'SARMAX (Exog. 4)',
    'sarmax_exog_1_3': 'SARMAX (Exog. 1&3)',
    'sarmax_exog_1_4': 'SARMAX (Exog. 1&4)',
    'lstm_ts42_single': 'LSTM (42 TimeSteps, Single)',
    'lstm_ts42_ensemble': 'LSTM (42 TimeSteps, Ensemble)',
    'lstm_ts42_recency': 'LSTM (42 TimeSteps, Weighted Ensemble)',
    'lstm_ts168_single': 'LSTM (168 TimeSteps, Single)',
    'lstm_ts168_ensemble': 'LSTM (168 TimeSteps, Ensemble)',
    'lstm_ts168_recency': 'LSTM (168 TimeSteps, Weighted Ensemble)',
    'lstm_feature_1_ts42_single': 'LSTM with Feature 1 (42 TimeSteps, Single)',
    'lstm_feature_1_ts42_ensemble': 'LSTM with Feature 1 (42 TimeSteps, Ensemble)',
    'lstm_feature_1_ts42_recency': 'LSTM with Feature 1 (42 TimeSteps, Weighted Ensemble)',
    'lstm_feature_1_ts168_single': 'LSTM with Feature 1 (168 TimeSteps, Single)',
    'lstm_feature_1_ts168_ensemble': 'LSTM with Feature 1 (168 TimeSteps, Ensemble)',
    'lstm_feature_1_ts168_recency': 'LSTM with Feature 1 (168 TimeSteps, Weighted Ensemble)',
    'lstm_feature_ts42_single': 'LSTM with Feature 4 (42 TimeSteps, Single)',
    'lstm_feature_ts42_ensemble': 'LSTM with Feature 4 (42 TimeSteps, Ensemble)',
    'lstm_feature_ts42_recency': 'LSTM with Feature 4 (42 TimeSteps, Weighted Ensemble)',
    'lstm_feature_ts168_single': 'LSTM with Feature 4 (168 TimeSteps, Single)',
    'lstm_feature_ts168_ensemble': 'LSTM with Feature 4 (168 TimeSteps, Ensemble)',
    'lstm_feature_ts168_recency': 'LSTM with Feature 4 (168 TimeSteps, Weighted Ensemble)',
    'lstm_features_1_3_ts42_single': 'LSTM with Features 1&3 (42 TimeSteps, Single)',
    'lstm_features_1_3_ts42_ensemble': 'LSTM with Features 1&3 (42 TimeSteps, Ensemble)',
    'lstm_features_1_3_ts42_recency': 'LSTM with Features 1&3 (42 TimeSteps, Weighted Ensemble)',
    'lstm_features_1_3_ts168_single': 'LSTM with Features 1&3 (168 TimeSteps, Single)',
    'lstm_features_1_3_ts168_ensemble': 'LSTM with Features 1&3 (168 TimeSteps, Ensemble)',
    'lstm_features_1_3_ts168_recency': 'LSTM with Features 1&3 (168 TimeSteps, Weighted Ensemble)',
    'lstm_features_1_4_ts42_single': 'LSTM with Features 1&4 (42 TimeSteps, Single)',
    'lstm_features_1_4_ts42_ensemble': 'LSTM with Features 1&4 (42 TimeSteps, Ensemble)',
    'lstm_features_1_4_ts42_recency': 'LSTM with Features 1&4 (42 TimeSteps, Weighted Ensemble)',
    'lstm_features_1_4_ts168_single': 'LSTM with Features 1&4 (168 TimeSteps, Single)',
    'lstm_features_1_4_ts168_ensemble': 'LSTM with Features 1&4 (168 TimeSteps, Ensemble)',
    'lstm_features_1_4_ts168_recency': 'LSTM with Features 1&4 (168 TimeSteps, Weighted Ensemble)',
    'bidirectional_lstm_ts42_single': 'BiLSTM (42 TimeSteps, Single)',
    'bidirectional_lstm_ts42_ensemble': 'BiLSTM (42 TimeSteps, Ensemble)',
    'bidirectional_lstm_ts42_recency': 'BiLSTM (42 TimeSteps, Weighted Ensemble)',
    'bidirectional_lstm_ts168_single': 'BiLSTM (168 TimeSteps, Single)',
    'bidirectional_lstm_ts168_ensemble': 'BiLSTM (168 TimeSteps, Ensemble)',
    'bidirectional_lstm_ts168_recency': 'BiLSTM (168 TimeSteps, Weighted Ensemble)',
}

In [ ]:
def find_info_of_highest(dataset, metric):
    """
    Finds and returns the information about the row with the highest value for a specified metric in the dataset.
    
    Parameters:
    -----------
    dataset : pandas.DataFrame
        The dataset to search through.
    metric : str
        The column name of the metric to find the maximum value for.
        
    Returns:
    --------
    dict
        A dictionary containing relevant information about the row with the highest metric value.
        Keys include: 'MODEL', 'MODEL_NAME', 'PRODUCT', 'PRODUCT_SIGN', and 'PRICE_TYPES'.
    """

    # Filter dataset to have only 12_16 for product and NEG for sign
    dataset = dataset[(dataset['PRODUCT'] == '12_16') & (dataset['PRODUCT_SIGN'] == 'NEG')]
    
    # Find the index of the row with the maximum value for the specified metric
    max_idx = dataset[metric].idxmax()
    
    # Extract the row with the highest metric value
    highest_row = dataset.loc[max_idx]

    # Create a dictionary with the relevant information from the highest-performing row
    highest = {
        'MODEL': highest_row['MODEL'],
        'MODEL_NAME': highest_row['MODEL_NAME'],
        'PRODUCT': highest_row['PRODUCT'],
        'PRODUCT_SIGN': highest_row['PRODUCT_SIGN'],
        'PRICE_TYPES': highest_row['PRICE_TYPE']
    }
    return highest

In [ ]:
# def find_info_of_highest(dataset, metric, top_n=3):
#     """
#     Find the top N models for each combination of product, product_sign, and price_type.
    
#     Parameters:
#     -----------
#     dataset : pandas.DataFrame
#         DataFrame containing model performance data
#     metric : str
#         Column name of the metric to optimize (e.g., 'REVENUE')
#     top_n : int, default=3
#         Number of top models to return for each combination
        
#     Returns:
#     --------
#     list of dict
#         List of dictionaries containing information about top performing models
#         for each product/product_sign/price_type combination
#     """
#     top_models = []
    
#     # Get unique combinations of product, product_sign, and price_type
#     combinations = dataset[['PRODUCT', 'PRODUCT_SIGN', 'PRICE_TYPE']].drop_duplicates()
    
#     for _, combo in combinations.iterrows():
#         product = combo['PRODUCT']
#         product_sign = combo['PRODUCT_SIGN']
#         price_type = combo['PRICE_TYPE']
        
#         # Filter dataset for this specific combination
#         subset = dataset[
#             (dataset['PRODUCT'] == product) & 
#             (dataset['PRODUCT_SIGN'] == product_sign) & 
#             (dataset['PRICE_TYPE'] == price_type)
#         ]
        
#         # Sort by metric in descending order and get top N
#         top_subset = subset.nlargest(top_n, metric)
        
#         # Convert each row to dictionary format
#         for _, row in top_subset.iterrows():
#             model_info = {
#                 'MODEL': row['MODEL'],
#                 'MODEL_NAME': row['MODEL_NAME'],
#                 'PRODUCT': row['PRODUCT'],
#                 'PRODUCT_SIGN': row['PRODUCT_SIGN'],
#                 'PRICE_TYPE': row['PRICE_TYPE'],
#                 'RANK': len(top_models) % top_n + 1,  # Rank within this combination (1, 2, 3)
#                 metric: row[metric]
#             }
#             top_models.append(model_info)

#     return pd.DataFrame(top_models)

In [ ]:
# best_ex_statistical = find_info_of_highest(ex_statistical, 'REVENUE')
# best_rw_statistical = find_info_of_highest(rw_statistical, 'REVENUE')

# best_ex_lstm_42 = find_info_of_highest(ex_lstm_42, 'REVENUE')
# best_ex_lstm_168 = find_info_of_highest(ex_lstm_168, 'REVENUE')

best_ex_statistical = find_info_of_highest(ex_statistical, 'COMBINED_SCORE')
best_rw_statistical = find_info_of_highest(rw_statistical, 'COMBINED_SCORE')

best_ex_lstm_42 = find_info_of_highest(ex_lstm_42, 'COMBINED_SCORE')
best_ex_lstm_168 = find_info_of_highest(ex_lstm_168, 'COMBINED_SCORE')

print(f"Best Expanding Window Statistical Model: {best_ex_statistical}")
print(f"Best Rolling Window Statistical Model: {best_rw_statistical}")
print(f"Best Expanding Window LSTM Model (42): {best_ex_lstm_42}")
print(f"Best Expanding Window LSTM Model (168): {best_ex_lstm_168}")

In [ ]:
# best_ex_statistical = find_info_of_highest(ex_statistical, 'REVENUE')
# best_rw_statistical = find_info_of_highest(rw_statistical, 'REVENUE')

# best_ex_lstm_42 = find_info_of_highest(ex_lstm_42, 'REVENUE')
# best_ex_lstm_168 = find_info_of_highest(ex_lstm_168, 'REVENUE')

# best_ex_statistical.to_csv(postprocessed_data_path / 'best_expanding_window_statistical_models.csv', index=False)
# best_rw_statistical.to_csv(postprocessed_data_path / 'best_rolling_window_statistical_models.csv', index=False)
# best_ex_lstm_42.to_csv(postprocessed_data_path / 'best_expanding_window_lstm_models_42.csv', index=False)
# best_ex_lstm_168.to_csv(postprocessed_data_path / 'best_expanding_window_lstm_models_168.csv', index=False)

# # display(best_ex_statistical)
# # display(best_rw_statistical)
# # display(best_ex_lstm_42)
# # display(best_ex_lstm_168)

In [ ]:
def find_and_read_file(model, product, product_sign, price_type, output_path):
    """
    Searches for a CSV file with a specific naming pattern and reads it into a pandas DataFrame.
    The function recursively searches through the specified output path to find a file that 
    exactly matches the generated pattern, and returns the content as a DataFrame with 'DATE' 
    column parsed as datetime and set as index.
    Parameters:
    -----------
    model : str
        Model name to include in the file pattern.
    product : str
        Product name to include in the file pattern.
    product_sign : str
        Product sign identifier to include in the file pattern.
    price_type : str
        Price type to include in the file pattern.
    output_path : str
        Directory path to search for the file.
    Returns:
    --------
    pandas.DataFrame or None
        DataFrame containing the file contents with DATE as index if found,
        None if the file doesn't exist or the output_path is invalid.
    Notes:
    ------
    The file naming pattern is: 
    '{model}_model_processed_results_{product}_{product_sign}_{price_type}.csv'
    """
    name_pattern = f'{model}_model_processed_results_{product}_{product_sign}_{price_type}.csv'

    # Walking through output_path to find the file
    if os.path.exists(output_path):
        for root, dirs, files in os.walk(output_path):
            for file in files:
                if file == name_pattern:  # Use exact match instead of substring match
                    full_path = os.path.join(root, file)
                    print(f"Found file at: {full_path}")
                    return pd.read_csv(full_path, parse_dates=['DATE'], index_col=['DATE'])
        
        # If we reach here, we haven't found the file anywhere in output_path
        print(f"File not found in any subdirectory: {name_pattern}")
        return None
    else:
        print(f"Output path does not exist: {output_path}")
        return None

In [ ]:
# Group models into categories for easier processing
model_groups = {
    'ROLLING_WINDOW_STATISTICAL': ['rw_arma_3m', 'rw_arma_6m', 'rw_arma_12m',
                                   'rw_armax_exog_1_3m', 'rw_armax_exog_1_6m', 'rw_armax_exog_1_12m',
                                   'rw_armax_3m', 'rw_armax_6m', 'rw_armax_12m',
                                   'rw_armax_exog_1_3_3m', 'rw_armax_exog_1_3_6m', 'rw_armax_exog_1_3_12m',
                                   'rw_armax_exog_1_4_3m', 'rw_armax_exog_1_4_6m', 'rw_armax_exog_1_4_12m',
                                   'rw_sarma_3m', 'rw_sarma_6m', 'rw_sarma_12m',
                                   'rw_sarmax_exog_1_3m', 'rw_sarmax_exog_1_6m', 'rw_sarmax_exog_1_12m',
                                #    'rw_sarmax_3m', 'rw_sarmax_6m', 'rw_sarmax_12m',
                                #   'rw_sarmax_exog_1_3_3m', 'rw_sarmax_exog_1_3_6m', 'rw_sarmax_exog_1_3_12m',
                                #   'rw_sarmax_exog_1_4_3m', 'rw_sarmax_exog_1_4_6m', 'rw_sarmax_exog_1_4_12m'
                                   ],
    'EXPANDING_WINDOW_STATISTICAL': ['arma', 'sarma', 
                                     'armax_exog_1', 'armax', 'armax_exog_1_3', 'armax_exog_1_4',
                                     'sarmax_exog_1', 'sarmax', 'sarmax_exog_1_3', 'sarmax_exog_1_4'],
    'EXPANDING_WINDOW_LSTM': ['lstm', 'bidirectional_lstm', 'lstm_feature_1', 'lstm_feature', 'lstm_features_1_3', 'lstm_features_1_4']
}

# Define price types to process
price_types = ['AVERAGE', 'MARGINAL']

# Iterate through each price type and model group to collect and organize results
for price_type in price_types:
    for model_group in model_groups:
        model_group_df = None
        
        # Process each model within the current model group
        for model in model_groups[model_group]:
            # Find and read the model results file
            model_df = find_and_read_file(
                model, 
                best_ex_statistical['PRODUCT'], 
                best_ex_statistical['PRODUCT_SIGN'], 
                price_type, 
                BASE_OUTPUT_PATH
            )
            
            if model_df is not None:
                # Convert DATE to datetime if it's a string column (safety check)
                if 'DATE' in model_df.columns:
                    model_df['DATE'] = pd.to_datetime(model_df['DATE'])
                    model_df.set_index('DATE', inplace=True)
                
                # Add model identifiers as columns
                model_df['MODEL'] = model
                model_df['MODEL_NAME'] = new_names.get(model, model)
                
                # Rename the prediction and error metric columns to include the model name for comparison
                model_df = model_df.rename(columns={'D+1': f'D+1_{model}',
                                                    'RMSE': f'RMSE_{model}',
                                                    'MAPE': f'MAPE_{model}',
                                                    'MAE': f'MAE_{model}'})
                
                if model_group_df is None:
                    # First model in the group, use as base dataframe
                    model_group_df = model_df[['ACTUAL_VALUE', 'MARGINAL_PRICE', 'MODEL', 'MODEL_NAME', 
                                            f'D+1_{model}', f'RMSE_{model}', f'MAPE_{model}', f'MAE_{model}']].copy()
                else:
                    # Join with the existing dataframe on index (DATE) to add this model's columns
                    model_group_df = model_group_df.join(model_df[[f'D+1_{model}', f'RMSE_{model}', f'MAPE_{model}', f'MAE_{model}']])
        
        # Save the combined results for this model group if data was found
        if model_group_df is not None:
            model_group_df.to_csv(
                postprocessed_data_path / f'{model_group.lower()}_{best_ex_statistical["PRODUCT"]}_{best_ex_statistical["PRODUCT_SIGN"]}_{price_type}_results.csv',
                index=True
            )

In [ ]:
# def plot_histogram(dataset, model, product, product_sign, price_type, output_path, rw=None):
#     """
#     Creates and saves a histogram visualization of MAPE (Mean Absolute Percentage Error) values
#     for energy price forecasting models.
#     The histogram shows the distribution of MAPE values for the specified model, with vertical lines
#     indicating median MAPE values for the current model and other models in the dataset for comparison.
#     Parameters
#     ----------
#     dataset : pandas.DataFrame
#         DataFrame containing MAPE values for different models.
#     model : str
#         The main model to visualize (e.g., 'LSTM', 'GRU', 'Naive').
#     product : str
#         Energy product type being modeled (e.g., 'electricity', 'gas').
#     product_sign : str
#         Direction indicator (e.g., 'buy', 'sell').
#     price_type : str
#         Type of price being analyzed (e.g., 'spot', 'forward').
#     output_path : pathlib.Path
#         Directory path where the output histogram will be saved.
#     rw : int or None, optional
#         Rolling window size filter. If provided, only includes MAPE columns
#         with this rolling window value.
#     Returns
#     -------
#     None
#         The function saves the histogram plot to the specified output path and closes the figure.
#     Notes
#     -----
#     - The histogram bins range from 0 to 100% in increments of 10%.
#     - The main model's median is shown with a solid blue line.
#     - Comparison models are shown with dashed lines in different colors.
#     - The legend is sorted by median MAPE values (ascending).
#     - Special handling for models with timestep indicators (ts42, ts168).
#     - Uses the 'new_names' dictionary for model name display formatting.
#     """
#     plt.figure(figsize=(12, 12))

#     # Determine if model contains special timestep indicators
#     model_display_name = new_names.get(model, model)
#     # Keep timestep info for title but not for legend
#     title_display_name = model_display_name
#     if "ts42" in model.lower():
#         title_display_name = f"{model_display_name} - TimeStep 42"
#     elif "ts168" in model.lower():
#         title_display_name = f"{model_display_name} - TimeStep 168"

#     if model == 'Naive':
#         # Create the histogram for the specified model's MAPE
#         ax = sns.histplot(dataset[f'MAPE'] * 100, bins=np.arange(0, 101, 10), 
#                         color='limegreen', alpha=0.7, stat='percent')
        
#         # Add vertical line for this model's median MAPE
#         model_median = dataset[f'MAPE'].median() * 100
#         plt.axvline(x=model_median, color='blue', linestyle='-', linewidth=2,
#                     label=f'{model_display_name}: {model_median:.2f}%')
        
#     else:
#         # Extract the relevant MAPE columns from the dataset based on rw parameter
#         if rw is not None:
#             # Filter columns that contain both 'MAPE_' and the rolling window value
#             mape_columns = [col for col in dataset.columns if 'MAPE_' in col and f'_{rw}' in col]
#         else:
#             # Extract all MAPE columns from the dataset
#             mape_columns = [col for col in dataset.columns if 'MAPE_' in col]
        
#         # Create the histogram for the specified model's MAPE
#         ax = sns.histplot(dataset[f'MAPE_{model}'] * 100, bins=np.arange(0, 101, 10), 
#                         color='limegreen', alpha=0.7, stat='percent')
        
#         # Plot MAPE for all models in different colors
#         colors = plt.cm.tab10.colors  # Use a colormap for different models
#         all_medians = []
        
#         for i, col in enumerate(mape_columns):
#             # Skip the current model since it's already plotted
#             if col == f'MAPE_{model}':
#                 continue
                
#             model_name = col.replace('MAPE_', '')
#             # Get the display name without timestep for legend
#             legend_display_name = new_names.get(model_name, model_name)
#             median_val = dataset[col].median() * 100
#             all_medians.append(median_val)
            
#             plt.axvline(x=median_val, color=colors[i % len(colors)], linestyle='--', linewidth=1.5,
#                     label=f'{legend_display_name}: {median_val:.2f}%')
        
#         # Add vertical line for this model's median MAPE
#         model_median = dataset[f'MAPE_{model}'].median() * 100
#         plt.axvline(x=model_median, color='blue', linestyle='-', linewidth=2,
#                     label=f'{model_display_name}: {model_median:.2f}%')
    
#     # Set title and labels - use title_display_name with timestep info
#     plt.title(f'{title_display_name} - {product} - {product_sign} - {price_type}', fontsize=20)
#     plt.xlabel('MAPE (%)', fontsize=16)
#     plt.ylabel('Frequency (%)', fontsize=16)
    
#     # Set axis limits to ensure zero point alignment
#     plt.xlim(0, 100)
#     plt.ylim(0, 100)
    
#     # Set ticks
#     plt.xticks(np.arange(0, 101, 10), fontsize=12)
#     plt.yticks(range(0, 101, 10), fontsize=12)
    
#     # Add grid lines
#     plt.grid(True, linestyle='--', alpha=0.7)
    
#     # Save the median values and label texts for sorting
#     median_labels = []
    
#     # Add this model's median to the list
#     median_labels.append((model_median, f'{model_display_name}: {model_median:.2f}%', 'blue', '-', 2))
    
#     if model != 'Naive':
#         # Add other models' medians
#         for i, col in enumerate(mape_columns):
#             if col == f'MAPE_{model}':
#                 continue
                
#             model_name = col.replace('MAPE_', '')
#             # Get the display name without timestep info for legend
#             legend_display_name = new_names.get(model_name, model_name)
#             median_val = dataset[col].median() * 100
#             median_labels.append((median_val, f'{legend_display_name}: {median_val:.2f}%', 
#                                 colors[i % len(colors)], '--', 1.5))
    
#     # Sort by median value (ascending)
#     median_labels.sort(key=lambda x: x[0])

#     # Create sorted legend handles and labels
#     handles = []
#     labels = []
    
#     for median_val, label, color, style, width in median_labels:
#         # Create a Line2D object for the legend
#         handle = mlines.Line2D([], [], color=color, linestyle=style, linewidth=width)
#         handles.append(handle)
#         labels.append(label)

#     # Add a note about lines representing medians
#     handles.append(mlines.Line2D([], [], color='gray', linestyle='-', linewidth=0))
#     labels.append("All lines represent median MAPE values")

#     # Add the legend with sorted items
#     plt.legend(handles=handles, labels=labels, fontsize=14, loc='best')
    
#     plt.tight_layout()
    
#     # Construct the filename
#     filename = f'{model.lower()}_{product}_{product_sign}_{price_type}_histogram.png'
        
#     plt.savefig(
#         output_path / filename,
#         dpi=300, bbox_inches='tight'
#     )
#     plt.close()

In [ ]:
def plot_histogram(dataset, model, product, product_sign, price_type, output_path, rw=None):
    """
    Creates and saves a histogram visualization of MAPE (Mean Absolute Percentage Error) values
    for energy price forecasting models.
    The histogram shows the distribution of MAPE values for the specified model, with vertical lines
    indicating median MAPE values for the current model and other models in the dataset for comparison.
    Parameters
    ----------
    dataset : pandas.DataFrame
        DataFrame containing MAPE values for different models.
    model : str
        The main model to visualize (e.g., 'LSTM', 'GRU', 'Naive').
    product : str
        Energy product type being modeled (e.g., 'electricity', 'gas').
    product_sign : str
        Direction indicator (e.g., 'buy', 'sell').
    price_type : str
        Type of price being analyzed (e.g., 'spot', 'forward').
    output_path : pathlib.Path
        Directory path where the output histogram will be saved.
    rw : int or None, optional
        Rolling window size filter. If provided, only includes MAPE columns
        with this rolling window value.
    Returns
    -------
    None
        The function saves the histogram plot to the specified output path and closes the figure.
    Notes
    -----
    - The histogram bins range from 0 to 100% in increments of 10%.
    - The main model's median is shown with a solid blue line.
    - Comparison models are shown with dashed lines in different colors.
    - The legend is sorted by median MAPE values (ascending).
    - Special handling for models with timestep indicators (ts42, ts168).
    - Uses the 'new_names' dictionary for model name display formatting.
    """
    plt.figure(figsize=(8, 8))

    # Determine if model contains special timestep indicators
    model_display_name = new_names.get(model, model)
    # Keep timestep info for title but not for legend
    title_display_name = model_display_name
    if "ts42" in model.lower():
        title_display_name = f"{model_display_name} - TimeStep 42"
    elif "ts168" in model.lower():
        title_display_name = f"{model_display_name} - TimeStep 168"

    if model == 'Naive':
        # Create the histogram for the specified model's MAE
        ax = sns.histplot(dataset[f'MAE'] * 100, bins=np.arange(0, 1001, 100), 
                        color='limegreen', alpha=0.7, stat='percent')

        # Add vertical line for this model's median MAE
        model_median = dataset[f'MAE'].median() * 100
        plt.axvline(x=model_median, color='blue', linestyle='-', linewidth=2,
                    label=f'{model_display_name}: {model_median:.2f}')
        
    else:
        # Extract the relevant MAE columns from the dataset based on rw parameter
        if rw is not None:
            # Filter columns that contain both 'MAE_' and the rolling window value
            mae_columns = [col for col in dataset.columns if 'MAE_' in col and f'_{rw}' in col]
        else:
            # Extract all MAE columns from the dataset
            mae_columns = [col for col in dataset.columns if 'MAE_' in col]

        # Create the histogram for the specified model's MAE
        ax = sns.histplot(dataset[f'MAE_{model}'] * 100, bins=np.arange(0, 1001, 100),
                        color='limegreen', alpha=0.7, stat='percent')

        # Plot MAE for all models in different colors
        colors = plt.cm.tab10.colors  # Use a colormap for different models
        all_medians = []
        
        for i, col in enumerate(mae_columns):
            # Skip the current model since it's already plotted
            if col == f'MAE_{model}':
                continue

            model_name = col.replace('MAE_', '')
            # Get the display name without timestep for legend
            legend_display_name = new_names.get(model_name, model_name)
            median_val = dataset[col].median() * 100
            all_medians.append(median_val)
            
            plt.axvline(x=median_val, color=colors[i % len(colors)], linestyle='--', linewidth=1.5,
                    label=f'{legend_display_name}: {median_val:.2f}')

        # Add vertical line for this model's median MAE
        model_median = dataset[f'MAE_{model}'].median() * 100
        plt.axvline(x=model_median, color='blue', linestyle='-', linewidth=2,
                    label=f'{model_display_name}: {model_median:.2f}')
    
    # Set title and labels - use title_display_name with timestep info
    plt.title(f'{title_display_name} - {product} - {product_sign} - {price_type}', fontsize=15, pad=20)
    plt.xlabel('MAE [(EUR/MW)/h]', fontsize=16)
    plt.ylabel('Frequency [%]', fontsize=16)
    
    # Set axis limits to ensure zero point alignment
    plt.xlim(0, 1000)
    plt.ylim(0, 100)
    
    # Set ticks
    plt.xticks(np.arange(0, 1001, 100), fontsize=12)
    plt.yticks(range(0, 101, 10), fontsize=12)
    
    # Add grid lines
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Save the median values and label texts for sorting
    median_labels = []
    
    # Add this model's median to the list
    median_labels.append((model_median, f'{model_display_name}: {model_median:.2f}', 'blue', '-', 2))
    
    if model != 'Naive':
        # Add other models' medians
        for i, col in enumerate(mae_columns):
            if col == f'MAE_{model}':
                continue
                
            model_name = col.replace('MAE_', '')
            # Get the display name without timestep info for legend
            legend_display_name = new_names.get(model_name, model_name)
            median_val = dataset[col].median() * 100
            median_labels.append((median_val, f'{legend_display_name}: {median_val:.2f}', 
                                colors[i % len(colors)], '--', 1.5))
    
    # Sort by median value (ascending)
    median_labels.sort(key=lambda x: x[0])

    # Create sorted legend handles and labels
    handles = []
    labels = []
    
    for median_val, label, color, style, width in median_labels:
        # Create a Line2D object for the legend
        handle = mlines.Line2D([], [], color=color, linestyle=style, linewidth=width)
        handles.append(handle)
        labels.append(label)

    # Add a note about lines representing medians
    handles.append(mlines.Line2D([], [], color='gray', linestyle='-', linewidth=0))
    labels.append("All lines represent median MAE values")

    # Add the legend with sorted items
    plt.legend(handles=handles, labels=labels, fontsize=12, loc='best')
    
    plt.tight_layout()
    
    # Construct the filename
    filename = f'{model.lower()}_{product}_{product_sign}_{price_type}_histogram.png'
        
    plt.savefig(
        output_path / filename,
        dpi=300, bbox_inches='tight'
    )
    plt.close()

In [ ]:
# Create the plots directory if it doesn't exist
if not os.path.exists(plot_output_path):
    os.makedirs(plot_output_path)

# Define model groups and their result file name patterns
model_files = {
    'expanding_window_statistical': best_ex_statistical,
    'rolling_window_statistical': best_rw_statistical,
    'expanding_window_lstm_42': best_ex_lstm_42,
    'expanding_window_lstm_168': best_ex_lstm_168
}

# Add a naive model case
model_files['naive'] = best_ex_statistical

for price_type in price_types:
    # Process and plot each model group
    for file_prefix, best_model_info in model_files.items():
        product = best_model_info['PRODUCT']
        product_sign = best_model_info['PRODUCT_SIGN']
        model = best_model_info['MODEL'] if file_prefix != 'naive' else 'Naive'
        
        # For the naive model, create and save results
        if file_prefix == 'naive':
            # Get base data from ex_statistical results
            results_path = postprocessed_data_path / f'expanding_window_statistical_{product}_{product_sign}_{price_type}_results.csv'
            
            if os.path.exists(results_path):
                base_data = pd.read_csv(results_path, parse_dates=['DATE'], index_col=['DATE'])
                
                # Create naive model (forecast = previous day's actual value)
                naive_results = base_data[['ACTUAL_VALUE', 'MARGINAL_PRICE']].copy()
                naive_results['D+1'] = naive_results['ACTUAL_VALUE'].shift(1)
                naive_results = naive_results.dropna()
                
                # Calculate error metrics
                naive_results['RMSE'] = np.sqrt((naive_results['D+1'] - naive_results['ACTUAL_VALUE'])**2)
                naive_results['MAPE'] = np.abs((naive_results['D+1'] - naive_results['ACTUAL_VALUE']) / 
                                            naive_results['ACTUAL_VALUE'])
                naive_results['MAE'] = np.abs(naive_results['D+1'] - naive_results['ACTUAL_VALUE'])
                
                # Save naive results
                naive_file = postprocessed_data_path / f'naive_{product}_{product_sign}_{price_type}_results.csv'
                naive_results.to_csv(naive_file, index=True)
                
                # Plot histogram for naive model
                plot_histogram(naive_results, 'Naive', product, product_sign, price_type, plot_output_path)
        else:
            # Load model results file
            results_path = postprocessed_data_path / f'{file_prefix}_{product}_{product_sign}_{price_type}_results.csv'
            
            if os.path.exists(results_path):
                results_df = pd.read_csv(results_path, parse_dates=['DATE'], index_col=['DATE'])
            
            # For rolling window models, create histograms for each window size
            if 'rolling_window' in file_prefix:
                # Create histograms for each rolling window size (3m, 6m, 12m)
                for window_size in ['3m', '6m', '12m']:
                    # Use the base model name without the window size suffix
                    base_model_name = model.replace('_12m', '').replace('_6m', '').replace('_3m', '')
                    # Create the full model name with the current window size
                    rw_model = f"{base_model_name}_{window_size}"
                    # Plot histogram for this specific rolling window size
                    plot_histogram(results_df, rw_model, product, product_sign, price_type, plot_output_path, window_size)
            else:
                # For other models, plot a single histogram
                plot_histogram(results_df, model, product, product_sign, price_type, plot_output_path)

In [ ]:
def plot_distribution(statistical_dataset, lstm_dataset, statistical_model, lstm_model, product, product_sign, price_type, output_path):
    """
    Plots the distribution of forecasts from statistical and LSTM models against actual values for the first 
    and last 3 months of the common time period in two separate graphs.
    Parameters:
    -----------
    statistical_dataset : pandas.DataFrame
        DataFrame containing the statistical model forecasts with a DatetimeIndex.
    lstm_dataset : pandas.DataFrame
        DataFrame containing the LSTM model forecasts with a DatetimeIndex.
    statistical_model : str
        Name of the statistical model used for forecasting.
    lstm_model : str
        Name of the LSTM model used for forecasting.
    product : str
        The product being forecasted (e.g., 'NCG', 'TTF').
    product_sign : str
        Sign or identifier for the product.
    price_type : str
        Type of price being analyzed (e.g., 'Bid', 'Ask').
    output_path : pathlib.Path
        Directory path where the generated plots will be saved.
    Notes:
    ------
    - Both plots have a fixed y-axis range from 0 to 30 with ticks every 5 units.
    - The plots use a 5-day interval for the x-axis.
    - The function saves two PNG files: one for the first 3 months and one for the last 3 months
      of the common date range between the two datasets.
    - The function uses a dictionary 'new_names' to map model names to display names in the plots.
    """
    # Get first 3 months and last 3 months of data (approximately 90 days)
    first_date_stat = statistical_dataset.index.min()
    last_date_stat = statistical_dataset.index.max()
    first_date_lstm = lstm_dataset.index.min()
    last_date_lstm = lstm_dataset.index.max()
    
    # Determine the common date range
    earliest_start = max(first_date_stat, first_date_lstm)
    latest_end = min(last_date_stat, last_date_lstm)
    
    # Calculate time periods
    three_months_from_start = earliest_start + pd.Timedelta(days=90)
    three_months_before_end = latest_end - pd.Timedelta(days=90)
    
    # Filter datasets for first and last 3 months
    stat_first = statistical_dataset[(statistical_dataset.index >= earliest_start) & 
                                    (statistical_dataset.index <= three_months_from_start)].copy()
    lstm_first = lstm_dataset[(lstm_dataset.index >= earliest_start) & 
                             (lstm_dataset.index <= three_months_from_start)].copy()
    
    stat_last = statistical_dataset[(statistical_dataset.index >= three_months_before_end) & 
                                   (statistical_dataset.index <= latest_end)].copy()
    lstm_last = lstm_dataset[(lstm_dataset.index >= three_months_before_end) & 
                            (lstm_dataset.index <= latest_end)].copy()
    
    # Reset indices to get DATE as column for plotting
    stat_first = stat_first.reset_index()
    lstm_first = lstm_first.reset_index()
    stat_last = stat_last.reset_index()
    lstm_last = lstm_last.reset_index()
    
    # Plot the first 3 months
    plt.figure(figsize=(24, 8))
    sns.lineplot(
        data=stat_first,
        x='DATE',
        y=f'D+1_{statistical_model}',
        label=f'{new_names.get(statistical_model, statistical_model)}',
        color='blue'
    )
    sns.lineplot(
        data=lstm_first,
        x='DATE',
        y=f'D+1_{lstm_model}',
        label=f'{new_names.get(lstm_model, lstm_model)}',
        color='orange'
    )
    sns.lineplot(
        data=lstm_first,  # We can use either dataset for actual values
        x='DATE',
        y='ACTUAL_VALUE',
        label='Actual Value',
        color='green'
    )
    plt.title(f'{new_names.get(statistical_model, statistical_model)} vs {new_names.get(lstm_model, lstm_model)} - {product} - {product_sign} - {price_type}', fontsize=16)
    plt.xlabel('Date', fontsize=14)
    plt.ylabel('Value [EUR/MW/h]', fontsize=14)
    plt.xticks(rotation=45)
    plt.ylim(0, 30)
    plt.yticks(range(0, 31, 5))
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)
    
    plt.tight_layout()
    plt.savefig(
        output_path / f'{statistical_model.lower()}_{lstm_model.lower()}_{product}_{product_sign}_{price_type}_first_3_months.png',
        dpi=300, bbox_inches='tight'
    )
    plt.close()
    
    # Plot the last 3 months
    plt.figure(figsize=(24, 8))
    sns.lineplot(
        data=stat_last,
        x='DATE',
        y=f'D+1_{statistical_model}',
        label=f'{new_names.get(statistical_model, statistical_model)}',
        color='blue'
    )
    sns.lineplot(
        data=lstm_last,
        x='DATE',
        y=f'D+1_{lstm_model}',
        label=f'{new_names.get(lstm_model, lstm_model)}',
        color='orange'
    )
    sns.lineplot(
        data=lstm_last,  # We can use either dataset for actual values
        x='DATE',
        y='ACTUAL_VALUE',
        label='Actual Value',
        color='green'
    )
    plt.title(f'{new_names.get(statistical_model, statistical_model)} vs {new_names.get(lstm_model, lstm_model)} - {product} - {product_sign} - {price_type}', fontsize=16)
    plt.xlabel('Date', fontsize=14)
    plt.ylabel('Value [EUR/MW/h]', fontsize=14)
    plt.xticks(rotation=45)
    plt.ylim(0, 30)
    plt.yticks(range(0, 31, 5))
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)
    
    plt.tight_layout()
    plt.savefig(
        output_path / f'{statistical_model.lower()}_{lstm_model.lower()}_{product}_{product_sign}_{price_type}_last_3_months.png',
        dpi=300, bbox_inches='tight'
    )
    plt.close()

In [ ]:
# Get product and product_sign from the best expanding window statistical model
product = best_ex_statistical['PRODUCT']
product_sign = best_ex_statistical['PRODUCT_SIGN']

# Loop through different price types (AVERAGE and MARGINAL)
for price_type in price_types:
    # Define paths to the CSV files containing model results
    # For statistical models
    ex_statistical_path = postprocessed_data_path / f'expanding_window_statistical_{product}_{product_sign}_{price_type}_results.csv'
    # For LSTM models with 42-hour timestep
    ex_lstm_42_path = postprocessed_data_path / f'expanding_window_lstm_42_{product}_{product_sign}_{price_type}_results.csv'

    # Call the plot_distribution function to create comparison charts
    # This function will create two plots:
    # 1. Comparing statistical vs LSTM forecasts for the first 3 months
    # 2. Comparing statistical vs LSTM forecasts for the last 3 months
    plot_distribution(
        # Load statistical model results with date as index
        pd.read_csv(ex_statistical_path, parse_dates=['DATE'], index_col=['DATE']),
        # Load LSTM model results with date as index
        pd.read_csv(ex_lstm_42_path, parse_dates=['DATE'], index_col=['DATE']),
        # Use the best statistical model based on revenue
        best_ex_statistical['MODEL'],
        # Use the best LSTM model with 42-hour timestep based on revenue
        best_ex_lstm_42['MODEL'],
        # Pass other parameters needed for file naming and chart titles
        product,
        product_sign,
        price_type,
        # Directory where plots will be saved
        plot_output_path
    )

In [16]:
def plot_models_comparison(dataset, price_type, metric, output_folder):
    """
    Creates separate comparison plots of top 3 performing models for NEG and POS product signs.
    This function visualizes the performance of the top 3 models from each model group 
    (EXPANDING_WINDOW_STATISTICAL, ROLLING_WINDOW_STATISTICAL, EXPANDING_WINDOW_LSTM) 
    for each specific product and product_sign combination, with model names displayed in a table below the plot.
    Parameters:
    -----------
    dataset : pandas.DataFrame
        The dataset containing model comparison data with columns:
        'PRICE_TYPE', 'MODEL_GROUP', 'MODEL_NAME', 'PRODUCT', 'PRODUCT_SIGN', 
        and the specified metric column.
    price_type : str
        The capacity price type to filter by (e.g., 'AVERAGE', 'MARGINAL').
    metric : str
        The performance metric to plot, such as 'REVENUE' or 'COMBINED_SCORE'.
        For 'REVENUE', higher values are better.
        For error metrics like 'COMBINED_SCORE', higher values are also better in normalized scores.
    output_folder : pathlib.Path
        The folder path where the plot image will be saved.
    Returns:
    --------
    None
        The function saves the plot as an image file and closes the plot.
    Notes:
    ------
    - Creates separate plots for NEG and POS product signs.
    - For each model group and product-sign combination, the top 3 performing models are selected.
    - Different colors represent different model groups, with different markers for ranking (1st, 2nd, 3rd).
    - Model names are displayed in a table below the plot.
    - For 'REVENUE' metric, y-axis ranges from 0 to 10,000 with 1,000 increments.
    - For 'COMBINED_SCORE' metric, y-axis is fixed between 0.99 and 1.0.
    - The plots are saved as PNG files with naming convention: 
      'model_comparison_{price_type}_{product_sign}_{metric}.png'
    """
    # Filter the dataset for the specified price type
    dataset = dataset.loc[dataset['PRICE_TYPE'] == price_type]

    # Define model groups and assign different colors to each group
    model_groups = dataset['MODEL_GROUP'].unique()
    group_colors = {
        'EXPANDING_WINDOW_STATISTICAL': 'blue',
        'ROLLING_WINDOW_STATISTICAL': 'green',
        'EXPANDING_WINDOW_LSTM': 'red',
    }
    
    # Define markers for different ranks
    rank_markers = {1: 'o', 2: 's', 3: '^'}  # circle, square, triangle
    rank_sizes = {1: 120, 2: 100, 3: 80}     # different sizes for visibility
    
    products = dataset['PRODUCT'].unique()
    product_signs = dataset['PRODUCT_SIGN'].unique()

    # Create separate plots for each product sign
    for product_sign in product_signs:
        # Create figure with subplot for plot and table
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), 
                                       gridspec_kw={'height_ratios': [3, 1]})
        
        # Filter dataset for this specific product sign
        sign_dataset = dataset[dataset['PRODUCT_SIGN'] == product_sign]
        
        # Create x-axis labels for products only (since we're plotting one sign at a time)
        x_labels = list(products)
        
        # Create a mapping of x-axis positions
        x_positions = {}
        for j, product in enumerate(products):
            x_positions[product] = j

        # For each product and model group, find the top 3 models
        top_models_data = []
        
        for product in products:
            for group in model_groups:
                # Filter data for this specific product, sign, and model group
                subset = sign_dataset[
                    (sign_dataset['PRODUCT'] == product) & 
                    (sign_dataset['MODEL_GROUP'] == group)
                ]
                
                if not subset.empty and metric in subset.columns:
                    # Find the top 3 models for this combination
                    if metric == 'REVENUE':
                        # For revenue, higher is better
                        top_3 = subset.nlargest(3, metric)
                    else:
                        # For error metrics like COMBINED_SCORE, higher is better in normalized scores
                        top_3 = subset.nlargest(3, metric)
                    
                    for rank, (_, row) in enumerate(top_3.iterrows(), 1):
                        top_models_data.append({
                            'PRODUCT': product,
                            'MODEL_GROUP': group,
                            'MODEL_NAME': row['MODEL_NAME'],
                            'RANK': rank,
                            metric: row[metric]
                        })
        
        # Convert to DataFrame for easier handling
        top_models_df = pd.DataFrame(top_models_data)
        
        # Track plotted groups for legend
        plotted_groups = set()
        plotted_ranks = set()
        
        # Plot data points for top models on the first subplot
        for _, row in top_models_df.iterrows():
            product = row['PRODUCT']
            model_group = row['MODEL_GROUP']
            model_name = row['MODEL_NAME']
            rank = row['RANK']
            value = row[metric]
            
            x_pos = x_positions[product]
            
            # Add slight offset based on rank to prevent overlapping
            x_offset = (rank - 2) * 0.1  # -0.1, 0, 0.1 for ranks 1, 2, 3
            
            # Plot with appropriate color for the model group and marker for rank
            ax1.scatter(x_pos + x_offset, value, 
                    s=rank_sizes[rank],
                    color=group_colors.get(model_group, 'blue'),
                    marker=rank_markers[rank],
                    alpha=0.7,
                    label=model_group if model_group not in plotted_groups else "",
                    edgecolors='black', linewidth=0.5)
            
            plotted_groups.add(model_group)
            plotted_ranks.add(rank)

        # Set appropriate title based on metric and product sign
        if metric == 'REVENUE':
            title = f'Revenue by Top 3 Models per Group and Block for {price_type} capacity prices ({product_sign})'
            ylabel = 'Revenue [EUR/MW/h]'
            # For revenue, use fixed y-ticks from 0 to 10 by 0.2
            ax1.set_yticks(np.arange(0, 20.1, 2))
        elif metric == 'COMBINED_SCORE':
            title = f'Combined Error Score by Top 3 Models per Group and Block for {price_type} capacity prices ({product_sign})'
            ylabel = 'Combined Score'
            # Set fixed y-range between 0.99 and 1.0 for combined score
            ax1.set_ylim(0.99, 1.0)
            # Create appropriate ticks within this range
            ax1.set_yticks(np.linspace(0.99, 1.0, 11))
        else:
            title = f'{metric} by Top 3 Models per Group and Block for {price_type} capacity prices ({product_sign})'
            ylabel = metric

        ax1.set_title(title, fontsize=16)
        ax1.set_xlabel('Time Block', fontsize=14)
        ax1.set_ylabel(ylabel, fontsize=14)
        ax1.set_xticks(range(len(x_labels)))
        ax1.set_xticklabels(x_labels, rotation=45)
        ax1.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Create combined legend for model groups and ranks
        legend_elements = []
        
        # Add model group legends
        for group in sorted(plotted_groups):
            legend_elements.append(
                mpatches.Patch(color=group_colors.get(group, 'blue'), 
                             label=group.replace('_', ' ').title())
            )
        
        # Add separator
        legend_elements.append(mpatches.Patch(color='white', label=''))
        
        # Add rank legends
        for rank in sorted(plotted_ranks):
            legend_elements.append(
                mlines.Line2D([0], [0], marker=rank_markers[rank], color='gray',
                            markerfacecolor='gray', markersize=8, linestyle='None',
                            label=f'{rank}{"st" if rank==1 else "nd" if rank==2 else "rd"} Best')
            )
        
        ax1.legend(handles=legend_elements, fontsize=10, loc='best', ncol=2)
        
        # Create table data for the second subplot (showing all top 3 models)
        table_data = []
        table_headers = ['Time Block']
        
        # Add headers for each model group (with rank indicators)
        for group in sorted(model_groups):
            group_name = group.replace('_', ' ').title()
            table_headers.extend([f'{group_name} (1st)', f'{group_name} (2nd)', f'{group_name} (3rd)'])
        
        # Create table rows
        for product in products:
            row = [product]
            for group in sorted(model_groups):
                for rank in [1, 2, 3]:
                    # Find the model for this product, group, and rank
                    model_info = top_models_df[
                        (top_models_df['PRODUCT'] == product) & 
                        (top_models_df['MODEL_GROUP'] == group) &
                        (top_models_df['RANK'] == rank)
                    ]
                    if not model_info.empty:
                        model_name = model_info.iloc[0]['MODEL_NAME']
                        metric_value = model_info.iloc[0][metric]
                        # Truncate long model names for table readability
                        short_name = model_name[:15] + '...' if len(model_name) > 18 else model_name
                        row.append(f"{short_name}\n({metric_value:.3f})")
                    else:
                        row.append("-")
            table_data.append(row)
        
        # Create the table
        ax2.axis('tight')
        ax2.axis('off')
        
        table = ax2.table(cellText=table_data,
                         colLabels=table_headers,
                         cellLoc='center',
                         loc='center',
                         bbox=[0, 0, 1, 1])
        
        # Style the table
        table.auto_set_font_size(False)
        table.set_fontsize(7)  # Smaller font due to more columns
        table.scale(1, 1.5)    # Make rows taller
        
        # Color the header row
        for i in range(len(table_headers)):
            table[(0, i)].set_facecolor('#40466e')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        # Color cells based on model group
        for i, row in enumerate(table_data):
            col_idx = 1  # Start after the Time Block column
            for group in sorted(model_groups):
                color = group_colors.get(group, 'lightgray')
                for rank in [1, 2, 3]:
                    if col_idx < len(table_headers):
                        table[(i + 1, col_idx)].set_facecolor(color)
                        # Different alpha values for different ranks
                        alpha = 0.6 if rank == 1 else 0.4 if rank == 2 else 0.2
                        table[(i + 1, col_idx)].set_alpha(alpha)
                    col_idx += 1
        
        plt.tight_layout()
        
        # Save the plot with appropriate filename based on metric and product sign
        metric_str = str(metric).lower()
        plt.savefig(output_folder / f'model_comparison_top3_{price_type}_{product_sign}_{metric_str}.png', 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        # Print summary of top models for this product sign
        print(f"\nTop 3 models for {price_type} capacity prices ({product_sign}) by {metric}:")
        print("=" * 100)
        for product in products:
            print(f"\n{product}:")
            for group in sorted(model_groups):
                group_models = top_models_df[
                    (top_models_df['PRODUCT'] == product) & 
                    (top_models_df['MODEL_GROUP'] == group)
                ].sort_values('RANK')
                
                if not group_models.empty:
                    print(f"  {group}:")
                    for _, model_row in group_models.iterrows():
                        rank_suffix = "st" if model_row['RANK']==1 else "nd" if model_row['RANK']==2 else "rd"
                        print(f"    {model_row['RANK']}{rank_suffix}: {model_row['MODEL_NAME']} ({metric}: {model_row[metric]:.4f})")

In [17]:
all_models_comparison_path = postprocessed_data_path / 'all_models_comparison.csv'
all_models_comparison = pd.read_csv(all_models_comparison_path)
plot_models_comparison(all_models_comparison, 'AVERAGE', 'REVENUE', plot_output_path)
plot_models_comparison(all_models_comparison, 'MARGINAL', 'REVENUE', plot_output_path)


Top 3 models for AVERAGE capacity prices (NEG) by REVENUE:

00_04:
  EXPANDING_WINDOW_LSTM:
    1st: LSTM with Features 1&4 (42 TimeSteps, Single) (REVENUE: 6.4050)
    2nd: LSTM with Features 1&4 (168 TimeSteps, Single) (REVENUE: 6.1492)
    3rd: LSTM with Features 1&4 (168 TimeSteps, Weighted Ensemble) (REVENUE: 6.1117)
  EXPANDING_WINDOW_STATISTICAL:
    1st: SARMAX (Exog. 1&3) (REVENUE: 5.5061)
    2nd: ARMAX (Exog. 1&3) (REVENUE: 5.4306)
    3rd: SARMAX (Exog. 4) (REVENUE: 5.1911)
  ROLLING_WINDOW_STATISTICAL:
    1st: ARMAX (Exog. 1&3) (3 Months) (REVENUE: 5.7350)
    2nd: ARMAX (Exog. 1&3) (12 Months) (REVENUE: 5.7308)
    3rd: ARMAX (Exog. 1&3) (6 Months) (REVENUE: 5.5105)

04_08:
  EXPANDING_WINDOW_LSTM:
    1st: LSTM with Feature 4 (168 TimeSteps, Single) (REVENUE: 5.9835)
    2nd: LSTM with Features 1&4 (168 TimeSteps, Single) (REVENUE: 5.9641)
    3rd: LSTM (42 TimeSteps, Weighted Ensemble) (REVENUE: 5.8810)
  EXPANDING_WINDOW_STATISTICAL:
    1st: ARMAX (Exog. 1&3) (REVEN